# Übungen – Block 9: Templates, Bedingungen und Handler

Die Übungen verwenden ein vorhandenes Inventory und Playbook-Projekt.

## Musterlösung

## Übung 1: Erstes Template

Erstellen Sie ein Jinja2-Template mit Hostname, Rolle und Port.

`templates/app.conf.j2`:

```jinja2
hostname={{ ansible_hostname }}
role={{ server_role }}
port={{ service_port }}
```

## Übung 2: Template verteilen

Verwenden Sie `template`, um die Datei auf eine Hostgruppe zu übertragen.

```yaml
- name: Konfiguration erzeugen
  ansible.builtin.template:
    src: app.conf.j2
    dest: /tmp/app.conf
```

## Übung 3: Bedingungen

Führen Sie einen Task nur auf Hosts mit einer bestimmten Rolle oder Distribution aus.

```yaml
- name: Nur auf Debian-Familie
  ansible.builtin.debug:
    msg: "Debian-basiertes System"
  when: ansible_os_family == "Debian"
```

## Übung 4: Loops

Legen Sie mehrere Verzeichnisse oder Dateien aus einer Liste heraus an.

```yaml
- name: Verzeichnisse anlegen
  ansible.builtin.file:
    path: "/tmp/{{ item }}"
    state: directory
  loop:
    - app
    - logs
    - data
```

## Übung 5: Handler

Lösen Sie bei Änderung einer Konfigurationsdatei einen Handler aus.

```yaml
tasks:
  - name: Konfiguration schreiben
    ansible.builtin.template:
      src: app.conf.j2
      dest: /tmp/app.conf
    notify: Restart demo

handlers:
  - name: Restart demo
    ansible.builtin.debug:
      msg: "Dienst würde neu gestartet"
```

## Übung 6: Idempotenz

Führen Sie das Playbook zweimal aus und prüfen Sie, ob der Handler beim zweiten Lauf erneut ausgeführt wird.

Beim ersten Lauf kann der Template-Task `changed` melden und den Handler auslösen. Bleibt die erzeugte Datei unverändert, sollte beim zweiten Lauf der Task `ok` melden und der Handler nicht erneut laufen.

## Abschlussübung

Kombinieren Sie Template, Variablen, Fact, `when`, `loop`, `notify` und Handler in einem Playbook.

```yaml
- name: Block 9 Abschluss
  hosts: all
  vars:
    service_port: 8080
    server_role: training
    dirs: [app, logs, data]

  tasks:
    - name: Verzeichnisse anlegen
      ansible.builtin.file:
        path: "/tmp/{{ item }}"
        state: directory
      loop: "{{ dirs }}"

    - name: Konfiguration erzeugen
      ansible.builtin.template:
        src: app.conf.j2
        dest: /tmp/app/app.conf
      notify: Show restart
      when: ansible_system == "Linux"

  handlers:
    - name: Show restart
      ansible.builtin.debug:
        msg: "Konfiguration geändert – Dienstaktion erforderlich"
```